# WorldWater Multi-Job Processing 
This notebook demonstrates how to use `MultiBackendJobManager` to run **monthly WorldWater water-extent processing** over a large area by splitting it into tiles and executing jobs in parallel.

The workflow is adapted from the openEO community example: [Visualising Multiple openEO Jobs](https://github.com/Open-EO/openeo-community-examples/blob/main/python/ManagingMultipleLargeScaleJobs/VisualisingMultipleOpeneoJobs.ipynb), with adjustments for WorldWater-specific parameters, monitoring, and raster merging.

## 1. Initialize Backend Connection and Job Manager
Connect to the backend and set up the MultiBackendJobManager to handle parallel job execution with 2 concurrent jobs per backend. Specify the output directory for storing job results and logs.

In [ ]:
from utils import *
from openeo.extra.job_management import MultiBackendJobManager
import openeo


output_dir = "test_multijob"

manager = MultiBackendJobManager(poll_sleep=60, root_dir=output_dir)
manager.add_backend("cdse", parallel_jobs=4, connection=openeo.connect(
    "https://openeo.dataspace.copernicus.eu/").authenticate_oidc())


Authenticated using refresh token.


## 2. Define Spatial Extent And Job Parameters
Set AOI bounds, time window, processing flags, and build the job table used by the job manager. Here we also split the AOI in a grid NxN, define my the grid_size parameter. 

In [ ]:
spatial_extent = {
    'west': 7.3604383130000306, 
    'east': 9.1599940380000362,
    'south': 55.8469481110000743,
    'north': 56.8437093290000348,
    'crs': 4326
}

grid_size = 25000
month_start="2024-01-01"
month_end="2024-02-01"
region="Temperate grassland"
threshold=75
cloud_cover=80
only_s1=False
use_sentinelhub=False
rgb_processing=False

job_file_path="jobs.csv"

job_df = create_job_df(spatial_extent, region, month_start, month_end, cloud_cover,
                       threshold, only_s1, use_sentinelhub, rgb_processing, output_dir=output_dir, grid_size=grid_size)


## Visualize the tiling grid
We use Plotly to create an interactive visualization of the spatial grid. This allows us to examine the layout of tiles across the area of interest and ensure that the grid aligns correctly with our region.

In [13]:

extents, geoms = split_extent(spatial_extent=spatial_extent, grid_size=grid_size)
grid_df = gpd.GeoDataFrame(geoms,columns=['geometry'],geometry='geometry',crs=4326)

centroids = grid_df.to_crs(3857).geometry.centroid.to_crs(4326)
center_lat, center_lon = centroids.y.mean(), centroids.x.mean()

grid_df["id"] = range(len(grid_df))

fig = px.choropleth_map(
    grid_df,
    geojson=grid_df.__geo_interface__,
    locations="id",
    color="id",
    map_style="carto-positron",
    center={"lat": center_lat, "lon": center_lon},
    zoom=8,
)

fig.update_layout(margin=dict(r=0,t=0,l=0,b=0))
fig.show()

## 3. Configure Job Parameters and Create Job Table
Here, we initate our job tracker jobs.csv from the earlier created dataframe `job_df`. This job tracker will be used to periodically obtain a status update the openEO jobs.

In [21]:
from openeo.extra.job_management import CsvJobDatabase
job_db = CsvJobDatabase(path=job_file_path)
if not job_db.exists():
    df = manager._normalize_df(job_df)
    job_db.persist(df)

## 5. Running the Jobs with MultiBackendJobManager
Finally, we run the jobs using MultiBackendJobManager, which allows us to manage multiple job executions across. As a standard-user, you can run 2 parallel jobs at any time.

Threading is applied to enable the visualization of job statuses while concurrently running openEO jobs. This approach allows the jobs to execute in parallel with the status updates, ensuring that the map is refreshed regularly without blocking job execution. In total there are two threads:

- Job Execution: The jobs are initiated using the job manager that runs in its own thread. This allows the jobs to be executed asynchronously.
- Visualization: At the same time, we continually check the job statuses from a CSV file (jobs.csv) and update the visualization using the plot_job_status function.


In [23]:
from plotly import offline
import time
from IPython.display import clear_output

colors = {
    "not_started": 'lightgrey', 
    "created": 'gold', 
    "queued": 'lightsteelblue', 
    "running": 'navy', 
    "finished": 'lime',
    "error": 'darkred',
    "skipped": 'darkorange',
    "start_failed": 'red',
    None: 'grey'  # Default color for any undefined status
}

# Start background scheduling/execution of jobs from the CSV database.
manager.start_job_thread(start_job=start_job, job_db=job_db)

while not manager._stop_thread:
    try:
        # Reload the latest status table on every refresh iteration.
        status_df = pd.read_csv(job_file_path)
        fig = plot_job_status(status_df=status_df, color_dict=colors)
        clear_output()
        offline.iplot(fig)

        # Stop monitoring thread once no active/pending statuses remain.
        if status_df['status'].isin(["not_started", "created", "queued", "running"]).sum() == 0:
            manager.stop_job_thread()

        # Pause to avoid excessive polling and API load.
        time.sleep(60)

    except KeyboardInterrupt:
        # Allow manual interruption in interactive runs.
        break

## Total credits spent 

In [24]:
import pandas as pd
jobs_df = pd.read_csv(job_file_path)
print('Total amound of credits:', jobs_df['costs'].sum())

Total amound of credits: 154.0


## Merge Tile Outputs
Collect all generated tile rasters for each date range and merge them into one output raster per period.

In [ ]:
import rasterio
from rasterio.merge import merge
import glob
files = glob.glob(os.path.join(output_dir, "job*\*tif"))

# Group tiles by date range and mosaic each group into one raster.
for date_range in {x.split('RGB_')[-1].split('.')[0] for x in files}:

    subset_files = [x for x in files if date_range in x]
    srcs = [rasterio.open(fp) for fp in subset_files]
    mosaic, transform = merge(srcs)

    # Reuse metadata from the first tile and update shape/georeferencing.
    meta = srcs[0].meta.copy()
    meta.update({
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": transform
    })

    out_folder = os.path.join(output_dir, f"WWT_{region}_{date_range}")
    os.makedirs(out_folder, exist_ok=True)
    prefix = "water_onlyS1" if only_s1 else "water"
    filename_prefix = f"{prefix}_{date_range}"

    out_path = os.path.join(out_folder, f"{filename_prefix}_{date_range}.tif")
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(mosaic)

    ######## FOR RGB ONLY ########
    # out_folder = os.path.join(output_dir, "RGB")
    # os.makedirs(out_folder, exist_ok=True)
    # out_path = os.path.join(out_folder, f"RGB_{date_range.split('_')[-1][:-1]}.tif")
    # with rasterio.open(out_path, "w", **meta) as dst:
    #     dst.write(mosaic)


    # Close all source handles to avoid locked files on Windows.
    for s in srcs:
        s.close()

